# Train Vertex Forecast for multiple scenarios, evaluate performance

In [1]:
import sys

# this repo
sys.path.append("..")
import env_config

# env vars
PROJECT_ID = env_config.PROJECT_ID
LOCATION = env_config.LOCATION
PREFIX = env_config.PREFIX
SERVICE_ACCOUNT = env_config.VERTEX_SA

print(f"PREFIX          : {PREFIX}")
print(f"PROJECT_ID      : {PROJECT_ID}")
print(f"LOCATION        : {LOCATION}")
print(f"SERVICE_ACCOUNT : {SERVICE_ACCOUNT}")

PREFIX          : vertex-forecast-v1
PROJECT_ID      : hybrid-vertex
LOCATION        : us-central1
SERVICE_ACCOUNT : 934903580331-compute@developer.gserviceaccount.com


In [2]:
! gcloud config set project $PROJECT_ID

Updated property [core/project].


### imports

In [21]:
import pandas as pd
import numpy as np
from time import sleep
from datetime import datetime, timedelta

import logging
logging.disable(logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

from google.cloud import aiplatform, bigquery

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

bq = bigquery.Client(project=PROJECT_ID)
aiplatform.init(project=PROJECT_ID, location=LOCATION)

TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
print(f"TIMESTAMP: {TIMESTAMP}")

TIMESTAMP: 20250402-111650


### env config

In [15]:
REGION = 'us-central1'
EXPERIMENT = 'automl-sdk-multi'
SERIES = 'applied-forecasting'

BQ_PROJECT = PROJECT_ID
BQ_DATASET = f"a_vf_multi_eval_{env_config.VERSION}".replace('-','_')
BQ_TABLE = 'forecast'

BUCKET = PROJECT_ID
BUCKET_URI = f"gs://{BUCKET}"
URI = f"{BUCKET_URI}/{SERIES}/{EXPERIMENT}/models"
DIR = f"temp/{EXPERIMENT}"

print(f"EXPERIMENT : {EXPERIMENT}")
print(f"SERIES     : {SERIES}")
print(f"BQ_DATASET : {BQ_DATASET}")
print(f"BUCKET     : {BUCKET}")
print(f"BUCKET_URI : {BUCKET_URI}")
print(f"URI        : {URI}")
print(f"DIR        : {DIR}")

EXPERIMENT : automl-sdk-multi
SERIES     : applied-forecasting
BQ_DATASET : a_vf_multi_eval_v1
BUCKET     : hybrid-vertex
BUCKET_URI : gs://hybrid-vertex
URI        : gs://hybrid-vertex/applied-forecasting/automl-sdk-multi/models
DIR        : temp/automl-sdk-multi


### Create BigQuery Dataset

In [8]:
ds = bigquery.Dataset(f"{PROJECT_ID}.{BQ_DATASET}")
ds.location = 'us' #REGION
ds.labels = {'series': f"{SERIES}", 'experiment': f'{EXPERIMENT}'}
ds = bq.create_dataset(dataset = ds, exists_ok = True)

In [9]:
ds.dataset_id

'a_vf_multi_eval_v1'

### AutoML config

In [5]:
# CUSTOMIZE
TARGET_COLUMN = 'num_trips'
TIME_COLUMN = 'starttime'
SERIES_COLUMN = 'start_station_name'
SPLIT_COLUMN = 'splits'
#COVARIATE_COLUMNS = ['avg_tripduration', 'pct_subscriber', 'ratio_gender', 'capacity'] # could be empty
COVARIATE_COLUMNS_ATTRIBUTES = []
COVARIATE_COLUMNS_KNOWN = ['capacity']
COVARIATE_COLUMNS_UNKNOWN = ['avg_tripduration', 'pct_subscriber', 'ratio_gender']

# the data preparation included preparing the data at this level
FORECAST_GRANULARITY = 'DAY'

FORECAST_HORIZON_LENGTH = 14

# the data preparation included setting this value for splits = TEST
FORECAST_TEST_LENGTH = 14
FORECAST_VALIDATE_LENGTH = 14

## Prepare data

In [11]:
BQ_SOURCE1 = 'bigquery-public-data.new_york.citibike_trips'
BQ_SOURCE2 = 'bigquery-public-data.new_york.citibike_stations'
viz_limit = 12

In [16]:
# CUSTOMIZE
query = f"""
CREATE OR REPLACE TABLE `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}_source` AS
WITH
    STATION AS (
        SELECT
           start_station_name,
           EXTRACT(DATE FROM starttime) AS starttime,
           COUNT(*) AS num_trips,
           AVG(tripduration) as avg_tripduration,
           COUNTIF(usertype='Subscriber')/COUNT(*) as pct_subscriber,
           SAFE_DIVIDE(COUNTIF(gender='male'), COUNTIF(gender!='male')) as ratio_gender
        FROM `{BQ_SOURCE1}`
        WHERE start_station_name LIKE '%Central Park%'
        GROUP BY start_station_name, starttime
    ),
    STATION_INFO AS (
        SELECT
            name,
            max(capacity) as capacity
        FROM `{BQ_SOURCE2}`
        WHERE name LIKE '%Central Park%'
        GROUP BY name
    )
SELECT * EXCEPT(name)
FROM STATION A
LEFT OUTER JOIN STATION_INFO B
ON A.start_station_name = B.name
ORDER BY start_station_name, starttime
"""
# print(query)

job = bq.query(query = query)
job.result()
(job.ended-job.started).total_seconds()

1.96

In [19]:
query = f"""
    CREATE OR REPLACE TABLE `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}_prepped` AS
    SELECT *,
       CASE
           WHEN {TIME_COLUMN} > DATE_SUB((SELECT MAX({TIME_COLUMN}) FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}_source`), INTERVAL {FORECAST_TEST_LENGTH} {FORECAST_GRANULARITY}) THEN "TEST"
           WHEN {TIME_COLUMN} > DATE_SUB((SELECT MAX({TIME_COLUMN}) FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}_source`), INTERVAL {FORECAST_TEST_LENGTH}+{FORECAST_VALIDATE_LENGTH} {FORECAST_GRANULARITY}) THEN "VALIDATE"
           ELSE "TRAIN"
       END AS splits
    FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}_source`
"""
job = bq.query(query)
job.result()
(job.ended-job.started).total_seconds()

1.906

### get `rawSeries` and key dates for splits

In [22]:
query = f"""
    WITH
        SPLIT AS (
            SELECT splits, min({TIME_COLUMN}) as mindate, max({TIME_COLUMN}) as maxdate
            FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}_prepped`
            GROUP BY {SPLIT_COLUMN}
        ),
        TRAIN AS (
            SELECT mindate as start_date
            FROM SPLIT
            WHERE {SPLIT_COLUMN} ='TRAIN'
        ),
        VAL AS (
            SELECT mindate as val_start
            FROM SPLIT
            WHERE {SPLIT_COLUMN} = 'VALIDATE'
        ),
        TEST AS (
            SELECT mindate as test_start, maxdate as end_date
            FROM SPLIT
            WHERE {SPLIT_COLUMN} = 'TEST'
        )
    SELECT * EXCEPT(pos) FROM
    (SELECT *, ROW_NUMBER() OVER() pos FROM TRAIN)
    JOIN (SELECT *, ROW_NUMBER() OVER() pos FROM VAL)
    USING (pos)
    JOIN (SELECT *, ROW_NUMBER() OVER() pos FROM TEST)
    USING (pos)
"""
keyDates = bq.query(query).to_dataframe()
keyDates

,start_date,val_start,test_start,end_date
0,2013-07-01,2016-09-03,2016-09-17,2016-09-30


In [23]:
query = f"""
    SELECT {SERIES_COLUMN}, {TIME_COLUMN}, {SPLIT_COLUMN}, {TARGET_COLUMN}
    FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}_prepped`
    ORDER by {SERIES_COLUMN}, {TIME_COLUMN}
"""
rawSeries = bq.query(query).to_dataframe()

### Plot Time Series data

In [ ]:
## TODO

# Create Forecast Model(s)

## Create/Recall Dataset

> Link to BigQuery Table

A Vertex AI Dataset is the input for Vertex AI Forecasting Jobs (AutoML, Seq2Seq, TFT). This creates a link between Vertex AI and the external data that does not copy the data from its source. Changes made to the source will apply to all models trained on the data starting at the time of the change.

* [Documentation](https://cloud.google.com/vertex-ai/docs/tabular-data/forecasting/create-dataset) for forecasting dataset
* Python SDK reference for [aiplatform.TimeSeriesDataset.create()](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.TimeSeriesDataset#google_cloud_aiplatform_TimeSeriesDataset_create)

In [27]:
if SERIES in [ds.display_name for ds in aiplatform.TimeSeriesDataset.list()]:
    dataset = aiplatform.TimeSeriesDataset.list(filter = f'display_name={SERIES}')[0]
else:
    dataset = aiplatform.TimeSeriesDataset.create(
        display_name = f'{SERIES}', 
        bq_source = f'bq://{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}_prepped',
        labels = {'series' : f'{SERIES}', 'experiment' : f'{EXPERIMENT}'}
    )

print(f'Created/Retrieve Dataset: {dataset.display_name}')

Created/Retrieve Dataset: applied-forecasting


In [28]:
dataset.column_names

['ratio_gender',
 'starttime',
 'num_trips',
 'start_station_name',
 'avg_tripduration',
 'pct_subscriber',
 'splits',
 'capacity']

## Train Forecasting Model with multiple Context Windows

> train multiple Vertex AI AutoML Forecasting Models while trying different values for context window

One of the parameters set when using Vertex AI Forecasting is the context window: how far back the model looks for patterns that impact a time point

* see documentation for this guide on [choosing a good value for the context window](https://cloud.google.com/vertex-ai/docs/tabular-data/bp-tabular#how_to_find_a_good_value_for_the_context_window). 
* Start with a context window equal to the forecast horizon (step 1). 
* The next suggestion is a context window that is double the forecast horizon (step 2). 
* It then suggest (step 3) trying larger multiples until no longer seeing improvements in the evaluation metrics

### Function to Create and Run AutoML Forecasting Jobs

Reference for [aiplatform.AutoMLForecastingTrainingJob](https://googleapis.dev/python/aiplatform/latest/aiplatform.html#google.cloud.aiplatform.AutoMLForecastingTrainingJob)

Reference for [aiplatform.AutoMLForecastingTrainingJob.run](https://googleapis.dev/python/aiplatform/latest/aiplatform.html#google.cloud.aiplatform.AutoMLForecastingTrainingJob.run)

In [24]:
def scenario(ds, cw):
    
    # model registry work
    modelmatch = aiplatform.Model.list(filter = f'display_name={SERIES}_{EXPERIMENT}_cw{cw} AND labels.series={SERIES} AND labels.experiment={EXPERIMENT} AND labels.cw={cw}')
    if modelmatch:
        print("There is an existing model with versions: ", [f'{m.version_id}' for m in modelmatch])
        parent = modelmatch[0].resource_name
    else:
        print("This is the first training for this model")
        parent = ''    
    
    # column specs
    column_specs = dict.fromkeys(
        list(set(dataset.column_names) - set([SPLIT_COLUMN, SERIES_COLUMN])), 'auto'
    )
    
    JOB_NAME = f"{EXPERIMENT}_eval_cw{cw}_{TIMESTAMP}".replace('-','_')
    print(f"JOB_NAME : {JOB_NAME}")
    
    # create automl forecasting job
    forecasting_job = aiplatform.AutoMLForecastingTrainingJob(
        display_name = JOB_NAME,
        optimization_objective = "minimize-rmse",
        column_specs = column_specs,
        labels = {'series' : f'{SERIES}', 'experiment' : f'{EXPERIMENT}', 'cw':f'{cw}'}
    )
    
    # run automl forecasting job: note sync=False and context window
    forecast = forecasting_job.run(
        # data parameters
        dataset = dataset,
        target_column = TARGET_COLUMN,
        time_column = TIME_COLUMN,
        time_series_identifier_column = SERIES_COLUMN,
        time_series_attribute_columns = COVARIATE_COLUMNS_ATTRIBUTES,
        unavailable_at_forecast_columns = [TARGET_COLUMN] + COVARIATE_COLUMNS_UNKNOWN,
        available_at_forecast_columns = [TIME_COLUMN] + COVARIATE_COLUMNS_KNOWN,
        predefined_split_column_name = SPLIT_COLUMN,

        # forecast parameters
        forecast_horizon = FORECAST_HORIZON_LENGTH,
        data_granularity_unit = FORECAST_GRANULARITY,
        data_granularity_count = 1,
        context_window = cw,
        holiday_regions = ['GLOBAL', 'NA', 'US'],

        hierarchy_group_columns = [],
        hierarchy_group_total_weight = 1.0,
        hierarchy_temporal_total_weight = 2.0,
        hierarchy_group_temporal_total_weight = 1.0,

        # output parameters
        export_evaluated_data_items = True,
        export_evaluated_data_items_bigquery_destination_uri = f"bq://{BQ_PROJECT}:{BQ_DATASET}:{EXPERIMENT}_eval_cw{cw}",
        export_evaluated_data_items_override_destination = True,

        # running parameters
        validation_options = "fail-pipeline",
        budget_milli_node_hours = 1000,

        # model parameters
        model_display_name = f"trained_model_cw{cw}_{env_config.VERSION}",
        model_labels = {'series' : f'{SERIES}', 'experiment' : f'{EXPERIMENT}', 'cw':f'{cw}'},
        model_id = f"vf_model_cw{cw}_{env_config.VERSION}",
        parent_model = parent,
        is_default_version = True,

        # session parameters: False means continue in local session, True waits and logs progress
        sync = False
    )
    
    return forecast

In [25]:
context_windows = [7, 21, 35, 42]

In [29]:
scenarios = [
    scenario(dataset, cw) for cw in context_windows
]

This is the first training for this model
JOB_NAME : automl_sdk_multi_eval_cw7_20250402_111650
This is the first training for this model
JOB_NAME : automl_sdk_multi_eval_cw21_20250402_111650
This is the first training for this model
JOB_NAME : automl_sdk_multi_eval_cw35_20250402_111650
This is the first training for this model
JOB_NAME : automl_sdk_multi_eval_cw42_20250402_111650


In [31]:
for s in scenarios:
    print(s.display_name, s.resource_name)
    print(f'Review the model in the Vertex AI Model Registry:\nhttps://console.cloud.google.com/vertex-ai/locations/{REGION}/models/{s.name}?project={PROJECT_ID}\n\n')

# Retrieve Test Data

> Make a list that contains dataframes of test data from each of the models, including those created by the prerequisite notebooks if they are present.

In [32]:
def test_query(context_window):
    
    if context_window == 14:
        CW_TEST_TABLE = f'automl-console_eval'
    elif context_window == 28:
        CW_TEST_TABLE = f'automl-python_eval'
    else:
        CW_TEST_TABLE = f'{EXPERIMENT}_eval_cw{context_window}'
    
    if CW_TEST_TABLE in [table.table_id for table in bq.list_tables(f'{BQ_PROJECT}.{BQ_DATASET}')]:
        query = f"""
            SELECT
                DATE({TIME_COLUMN}) as {TIME_COLUMN},
                DATE(predicted_on_{TIME_COLUMN}) as predicted_on_{TIME_COLUMN},
                CAST({TARGET_COLUMN} as INT64) AS {TARGET_COLUMN},
                {SERIES_COLUMN},
                predicted_{TARGET_COLUMN}.value as predicted_{TARGET_COLUMN}
            FROM `{BQ_PROJECT}.{BQ_DATASET}.{CW_TEST_TABLE}`
            WHERE {TIME_COLUMN} = predicted_on_{TIME_COLUMN}
            ORDER BY {SERIES_COLUMN}, {TIME_COLUMN}
        """
        return (context_window, CW_TEST_TABLE, bq.query(query = query).to_dataframe())
    else:
        return

test_predictions = [test_query(context_window) for context_window in context_windows + [14, 28]]

In [33]:
test_predictions[0]

## Review Custom Metrics with SQL

In [35]:
def customMetrics_query(cw, CW_TEST_TABLE):
    query = f"""
    WITH
        FORECASTS AS (
            SELECT
                DATE({TIME_COLUMN}) as {TIME_COLUMN},
                DATE(predicted_on_{TIME_COLUMN}) as predicted_on_{TIME_COLUMN},
                CAST({TARGET_COLUMN} as INT64) AS {TARGET_COLUMN},
                {SERIES_COLUMN},
                predicted_{TARGET_COLUMN}.value as predicted_{TARGET_COLUMN}
            FROM `{BQ_PROJECT}.{BQ_DATASET}.{CW_TEST_TABLE}`
            WHERE {TIME_COLUMN} = predicted_on_{TIME_COLUMN}
        ),
        DIFFS AS (
            SELECT 
                {SERIES_COLUMN},
                {TIME_COLUMN},
                'forecast' as time_series_type,
                predicted_{TARGET_COLUMN} as forecast_value,
                {TARGET_COLUMN} as actual_value,
                ({TARGET_COLUMN} - predicted_{TARGET_COLUMN}) as diff
            FROM FORECASTS    
        )
    SELECT
        start_station_name,
        time_series_type, 
        AVG(SAFE_DIVIDE(ABS(diff), actual_value)) as MAPE,
        AVG(ABS(diff)) as MAE,
        SAFE_DIVIDE(SUM(ABS(diff)), SUM(actual_value)) as pMAE,
        AVG(POW(diff, 2)) as MSE,
        SQRT(AVG(POW(diff, 2))) as RMSE,
        SAFE_DIVIDE(SQRT(AVG(POW(diff, 2))), AVG(actual_value)) as pRMSE
    FROM DIFFS
    GROUP BY
        {SERIES_COLUMN},
        time_series_type
    ORDER BY
        {SERIES_COLUMN},
        time_series_type    
    """
    return (cw, CW_TEST_TABLE, bq.query(query = query).to_dataframe())

In [ ]:
customMetrics = [customMetrics_query(cw[0], cw[1]) for cw in test_predictions if cw]

[cm[2].insert(1, 'context_window', cm[0]) for cm in customMetrics]

customMetrics = pd.concat([cm[2] for cm in customMetrics]).sort_values(by = [SERIES_COLUMN, 'context_window'])
customMetrics

In [ ]:
customMetrics.head(10)

## Overall Metrics

In [ ]:
def customMetricsOverall_query(cw, CW_TEST_TABLE):
    query = f"""
    WITH
        FORECASTS AS (
            SELECT
                DATE({TIME_COLUMN}) as {TIME_COLUMN},
                DATE(predicted_on_{TIME_COLUMN}) as predicted_on_{TIME_COLUMN},
                CAST({TARGET_COLUMN} as INT64) AS {TARGET_COLUMN},
                {SERIES_COLUMN},
                predicted_{TARGET_COLUMN}.value as predicted_{TARGET_COLUMN}
            FROM `{BQ_PROJECT}.{BQ_DATASET}.{CW_TEST_TABLE}`
            WHERE {TIME_COLUMN} = predicted_on_{TIME_COLUMN}
        ),
        DIFFS AS (
            SELECT 
                {SERIES_COLUMN},
                {TIME_COLUMN},
                'forecast' as time_series_type,
                predicted_{TARGET_COLUMN} as forecast_value,
                {TARGET_COLUMN} as actual_value,
                ({TARGET_COLUMN} - predicted_{TARGET_COLUMN}) as diff
            FROM FORECASTS    
        )
    SELECT
        #start_station_name,
        time_series_type, 
        AVG(SAFE_DIVIDE(ABS(diff), actual_value)) as MAPE,
        AVG(ABS(diff)) as MAE,
        SAFE_DIVIDE(SUM(ABS(diff)), SUM(actual_value)) as pMAE,
        AVG(POW(diff, 2)) as MSE,
        SQRT(AVG(POW(diff, 2))) as RMSE,
        SAFE_DIVIDE(SQRT(AVG(POW(diff, 2))), AVG(actual_value)) as pRMSE
    FROM DIFFS
    GROUP BY
        #{SERIES_COLUMN},
        time_series_type
    ORDER BY
        #{SERIES_COLUMN},
        time_series_type    
    """
    return (cw, CW_TEST_TABLE, bq.query(query = query).to_dataframe())

In [ ]:
customMetricsOverall = [customMetricsOverall_query(cw[0], cw[1]) for cw in test_predictions if cw]

[cm[2].insert(1, 'context_window', cm[0]) for cm in customMetricsOverall]

customMetricsOverall = pd.concat([cm[2] for cm in customMetricsOverall]).sort_values(by = ['context_window'])

customMetricsOverall

## What is The "Best" Context Window?

In [ ]:
customMetricsOverall[customMetricsOverall['pMAE'] == customMetricsOverall['pMAE'].min()].reset_index()

## What is the best context window for each value of SERIES_COLUMN?

In [ ]:
bestCW = customMetrics[customMetrics['pMAE'] == customMetrics.groupby([SERIES_COLUMN])['pMAE'].transform('min')].reset_index()
bestCW

# Get Forecasted Values for Future Horizon

Use a batch prediction job with the resulting forecasting model to get predicted forecast for the future horizon.

The requirement for getting batch predictions from a Vertex AI forecasting model are covered [here](https://cloud.google.com/vertex-ai/docs/tabular-data/forecasting/get-predictions).

In [ ]:
def horizon_input_tables(context_window):

    # get the largest context window used in this series so far:
    #context_window = customMetricsOverall['context_window'].max()
    CW_HORIZON_INPUT_TABLE = f'{EXPERIMENT}_horizon_input_cw{context_window}' 


    query_a = ""
    query_b = ""
    for v in COVARIATE_COLUMNS_KNOWN + COVARIATE_COLUMNS_UNKNOWN + COVARIATE_COLUMNS_ATTRIBUTES:
        query_a += f""",
                LAST_VALUE({v} IGNORE NULLS) OVER (PARTITION BY {SERIES_COLUMN} ORDER BY {TIME_COLUMN} ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as {v}"""
        if v not in COVARIATE_COLUMNS_ATTRIBUTES:
            query_b += f""",
            CASE WHEN {TIME_COLUMN} > (SELECT MAX({TIME_COLUMN}) FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`) THEN NULL ELSE {v} END AS {v}"""
        else:
            query_b += f""",
            {v}"""

    query = f"""
    CREATE OR REPLACE TABLE `{BQ_PROJECT}.{BQ_DATASET}.{CW_HORIZON_INPUT_TABLE}` AS
    WITH
        DATELIST AS (
            SELECT *
            FROM (SELECT DISTINCT {SERIES_COLUMN} FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`) A
            CROSS JOIN (SELECT * 
                        FROM UNNEST(GENERATE_DATE_ARRAY(
                                        DATE_SUB((SELECT MAX({TIME_COLUMN}) FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`), INTERVAL {context_window-1} DAY),
                                        DATE_ADD((SELECT MAX({TIME_COLUMN}) FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`), INTERVAL {FORECAST_HORIZON_LENGTH} DAY),
                                        INTERVAL 1 DAY
                                    )
                                ) AS {TIME_COLUMN}
                        ) B
        ),
        ADDTARGET AS (
            SELECT *
            FROM DATELIST
            LEFT OUTER JOIN (SELECT * FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`)
            USING ({SERIES_COLUMN}, {TIME_COLUMN})
            ORDER BY {SERIES_COLUMN}, {TIME_COLUMN}
        ),
        LOCF AS (
            SELECT {SERIES_COLUMN}, {TIME_COLUMN},
            LAST_VALUE({TARGET_COLUMN} IGNORE NULLS) OVER (PARTITION BY {SERIES_COLUMN} ORDER BY {TIME_COLUMN} ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as {TARGET_COLUMN}
            {query_a}
            FROM ADDTARGET
        )
    SELECT {SERIES_COLUMN}, {TIME_COLUMN},
        CASE
            WHEN {TIME_COLUMN} > (SELECT MAX({TIME_COLUMN}) FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`) THEN NULL
            ELSE {TARGET_COLUMN}
        END AS {TARGET_COLUMN}
        {query_b}
    FROM LOCF
    ORDER BY {SERIES_COLUMN}, {TIME_COLUMN}
    """
    job = bq.query(query = query)
    job.result()
    return (CW_HORIZON_INPUT_TABLE, job.state)
    
horizon_input_table_jobs = [horizon_input_tables(cw) for cw in context_windows]
horizon_input_table_jobs

## Batch Prediction

Request the batch prediction directly using the batch_prediction() method for the model - SDK Reference.

In [ ]:
def batch_creator(forecast):
    batchjob = forecast.batch_predict(
        job_display_name = f"{SERIES}_{EXPERIMENT}_cw{forecast.labels['cw']}_{TIMESTAMP}",
        bigquery_source = f"bq://{BQ_PROJECT}.{BQ_DATASET}.{EXPERIMENT}_horizon_input_cw{forecast.labels['cw']}",
        bigquery_destination_prefix = f"bq://{BQ_PROJECT}.{BQ_DATASET}",
        sync = False
    )
    return batchjob

batchjobs = [batch_creator(forecast) for forecast in scenarios]

In [ ]:
for j, job in enumerate(batchjobs):
    print(job.display_name, 'Completed with status:', job.state.name)

In [ ]:
[batchjob.output_info.bigquery_output_table for batchjob in batchjobs]

## Process Predicted Forecast

In [ ]:
def horizon_output_tables(context_window, batchjob_index):
    
    if context_window == 14:
        CW_HORIZON_OUTPUT_TABLE = f'automl-console_horizon_output'
    elif context_window == 28:
        CW_HORIZON_OUTPUT_TABLE = f'automl-python_horizon_output'
    else:
        CW_HORIZON_OUTPUT_TABLE = f'{EXPERIMENT}_horizon_output_cw{context_window}'    
    
        query = f"""
            CREATE OR REPLACE TABLE `{BQ_PROJECT}.{BQ_DATASET}.{CW_HORIZON_OUTPUT_TABLE}` AS
            SELECT {SERIES_COLUMN}, DATE({TIME_COLUMN}) as {TIME_COLUMN}, predicted_{TARGET_COLUMN}.value as predicted_{TARGET_COLUMN}
            FROM `{BQ_PROJECT}.{BQ_DATASET}.{batchjobs[batchjob_index].output_info.bigquery_output_table}`
            ORDER BY {SERIES_COLUMN}, {TIME_COLUMN}
        """
        job = bq.query(query = query)
        job.result()
    
    if CW_HORIZON_OUTPUT_TABLE in [table.table_id for table in bq.list_tables(f'{BQ_PROJECT}.{BQ_DATASET}')]:
        query = f"""
            SELECT *
            FROM `{BQ_PROJECT}.{BQ_DATASET}.{CW_HORIZON_OUTPUT_TABLE}`
            ORDER BY {SERIES_COLUMN}, {TIME_COLUMN}
        """    
        return (context_window, CW_HORIZON_OUTPUT_TABLE, bq.query(query = query).to_dataframe())
    else:
        return

In [ ]:

horizon_predictions = [
    horizon_output_tables(context_window, batchjob_index) for batchjob_index, context_window in enumerate(context_windows + [14, 28])
]

horizon_predictions[0]

For each `SERIES_COLUMN`:
* lookup best context window in bestCW
* plot test forecast from test_predictions
  * has tuples of (context window, BQ TABLE NAME, dataframe[SERIES_COLUMN, prediction_{TARGET_COLUMN}])
* plot horizon forecast from horizon_predictions
  * has tuples of (context window, BQ TABLE NAME, dataframe[SERIES_COLUMN, predicted_{TARGET_COLUMN}])

In [ ]:
s = 'Central Park S & 6 Ave'

In [ ]:
local_cw = bestCW[bestCW[SERIES_COLUMN] == s]['context_window'].iloc[0]
local_cw

In [ ]:
local_test = [test_prediction[2] for tp, test_prediction in enumerate(test_predictions) if test_prediction[0] == local_cw][0]
local_test = local_test[local_test[SERIES_COLUMN] == s]
local_test

In [ ]:
local_horizon = [horizon_prediction[2] for tp, horizon_prediction in enumerate(horizon_predictions) if horizon_prediction[0] == local_cw][0]
local_horizon = local_horizon[local_horizon[SERIES_COLUMN] == s]
local_horizon

# Visualize The Time Series With Forecast

In [ ]:
# NA values in Pandas will not convert to JSON which Plotly uses:
rawSeries = rawSeries.fillna(np.nan).replace([np.nan], [None])

# create a figure:
fig = go.Figure()

# get a list of colors to use:
colors = px.colors.qualitative.Plotly

# list of columns to plot over time : target and covariates
variables = [TARGET_COLUMN] #+ COVARIATE_COLUMNS

# create dropdown/button to toggle series
buttons = []
b = 0 # default button index

# iterate through series:
series = rawSeries[SERIES_COLUMN].unique().tolist()[0:viz_limit]
for s in series:  
    ff = 0  
    # iterate trhough columns
    for y, v in enumerate(variables):
        fig.add_trace(
            go.Scatter(
                x = rawSeries[rawSeries[SERIES_COLUMN]==s][TIME_COLUMN],
                y = rawSeries[rawSeries[SERIES_COLUMN]==s][v],
                name = f'{v}',
                text = rawSeries[rawSeries[SERIES_COLUMN]==s][v],
                yaxis = f"y{y+1}",
                hoverinfo='name+x+text',
                line = {'width': 0.5},
                marker = {'size': 8},
                mode = 'lines+markers',
                showlegend = False,
                visible = (b==0) # make a series visible as default: this uses the first series
            )
        )
        if y == 0: # add the forecast
            # get the best context window avaiable for this series (s)
            local_cw = bestCW[bestCW[SERIES_COLUMN] == s]['context_window'].iloc[0]
            # get the test_predictions for this series (s) and best context window
            local_test = [test_prediction[2] for tp, test_prediction in enumerate(test_predictions) if test_prediction[0] == local_cw][0]
            local_test = local_test[local_test[SERIES_COLUMN] == s]
            # get the horizon_predictions for this series (s) and best context window
            local_horizon = [horizon_prediction[2] for tp, horizon_prediction in enumerate(horizon_predictions) if horizon_prediction[0] == local_cw][0]
            local_horizon = local_horizon[local_horizon[SERIES_COLUMN] == s]
            
            # add the forecast fit: test
            ff += 1
            fig.add_trace(
                go.Scatter(
                    x = local_test[TIME_COLUMN],
                    y = local_test[f'predicted_{TARGET_COLUMN}'],
                    name = f'Fit (cw = {local_cw}): {v}',
                    text = local_test[f'predicted_{TARGET_COLUMN}'],
                    yaxis = f"y{y+1}",
                    hoverinfo='name+x+text',
                    line = {'width': 2, 'color': 'rgb(255,234,0)'},
                    mode = 'lines',
                    showlegend = False,
                    visible = (b==0) # make a series visible as default: this uses the first series
                )
            )            
            
            # add the forecast fit: horizon
            ff += 1
            fig.add_trace(
                go.Scatter(
                    x = local_horizon[TIME_COLUMN],
                    y = local_horizon[f'predicted_{TARGET_COLUMN}'],
                    name = f'Fit (cw = {local_cw}): {v}',
                    text = local_horizon[f'predicted_{TARGET_COLUMN}'],
                    yaxis = f"y{y+1}",
                    hoverinfo='name+x+text',
                    line = {'width': 2, 'color': 'rgb(255,234,0)'},
                    mode = 'lines',
                    showlegend = False,
                    visible = (b==0) # make a series visible as default: this uses the first series
                )
            )
    
    # which button to show:
    #ff = 2 # count of forecast related traces add to each series
    which_buttons = [False] * len(series) * (len(variables) + ff)
    which_buttons[b * (len(variables) +ff):(b+1)*(len(variables) + ff)] = [True] * (len(variables) + ff)

    # create button for series:
    button = dict(
        label = s,
        method = 'update',
        args = [{'visible': which_buttons}]
    )
    buttons.append(button)
    b += 1

# add split regions: training
fig.add_shape(
    fillcolor = 'rgba(0, 255, 0, 0.2)',
    line = {'width': 0},
    type = 'rect',
    x0 = keyDates['start_date'][0],
    x1 = keyDates['val_start'][0],
    xref = 'x',
    y0 = 0,
    y1 = 1,
    yref = 'paper'
)
fig.add_annotation(
    x = keyDates['val_start'][0] - (keyDates['test_start'][0]-keyDates['val_start'][0])/2,
    y = 0,
    ax = 0, ay = 0,
    yref = 'y1',
    xref = 'x',
    text = 'Training',
    yanchor = 'bottom'
)

# add split regions: validation
fig.add_shape(
    fillcolor = 'rgba(255, 255, 0, 0.2)',
    line = {'width': 0},
    type = 'rect',
    x0 = keyDates['val_start'][0],
    x1 = keyDates['test_start'][0],
    xref = 'x',
    y0 = 0,
    y1 = 1,
    yref = 'paper'
)
fig.add_annotation(
    x = keyDates['val_start'][0] + (keyDates['test_start'][0]-keyDates['val_start'][0])/2,
    y = 0,
    ax = 0, ay = 0,
    yref = 'y1',
    xref = 'x',
    text = 'Validation',
    yanchor = 'bottom'
)

# add split regions: test
fig.add_shape(
    fillcolor = 'rgba(0, 0, 255, 0.2)',
    line = {'width': 0},
    type = 'rect',
    x0 = keyDates['test_start'][0],
    x1 = keyDates['end_date'][0],
    xref = 'x',
    y0 = 0,
    y1 = 1,
    yref = 'paper'
)
fig.add_annotation(
    x = keyDates['test_start'][0] + (keyDates['end_date'][0]-keyDates['test_start'][0])/2,
    y = 0,
    ax = 0, ay = 0,
    yref = 'y1',
    xref = 'x',
    text = 'Test',
    yanchor = 'bottom'
)

# add split regions: horizon
fig.add_shape(
    fillcolor = 'rgba(255, 255, 255, 0.2)',
    line = {'width': 0},
    type = 'rect',
    x0 = keyDates['end_date'][0],
    x1 = keyDates['end_date'][0]+timedelta(days = FORECAST_HORIZON_LENGTH),
    xref = 'x',
    y0 = 0,
    y1 = 1,
    yref = 'paper'
)
fig.add_annotation(
    x = keyDates['end_date'][0] + (keyDates['end_date'][0]-keyDates['test_start'][0])/2,
    y = 0,
    ax = 0, ay = 0,
    yref = 'y1',
    xref = 'x',
    text = 'Horizon',
    yanchor = 'bottom'
)

# configure axes layout:
layout = dict(
    xaxis =  dict(
        range = [keyDates['end_date'][0] - 2*(keyDates['end_date'][0] - keyDates['val_start'][0]), keyDates['end_date'][0]+timedelta(days = FORECAST_HORIZON_LENGTH)],
        rangeslider = dict(
            autorange = True,
            range = [keyDates['start_date'][0], keyDates['end_date'][0]+timedelta(days = FORECAST_HORIZON_LENGTH)]
        ),
        type = 'date'
    )
)
for v, variable in enumerate(variables):
    layout[f'yaxis{v+1}'] = dict(
        anchor = 'x',
        domain = [v*(1/len(variables)), (v+1)*(1/len(variables))],
        autorange = True,
        mirror = True,
        autoshift = True,
        title = dict(text = variable, standoff = 10 + 20 * (v % 2), font = dict(color = colors[v])),
        tickfont = dict(color = colors[v]),
        tickmode = 'auto',
        linecolor = colors[v],
        linewidth = 4,
        showline = True,
        side = 'right',
        type = 'linear',
        zeroline = False
    )

# final update of display before rendering
fig.update_layout(
    layout,
    title = 'Time Series Plots:',
    dragmode="zoom",
    hovermode="x",
    legend=dict(traceorder="reversed"),
    height=600,
    template="plotly_white",
    margin=dict(
        t=100,
        b=100
    ),
    updatemenus = [
        dict(
            buttons = buttons,
            type = 'dropdown',
            direction = 'down',
            x = 1,
            y = 1.2,
            showactive = True
        )
    ]
)

# render the interactive plot:
fig.show()